## Example 1: A client pointed at a region

In [ ]:
import importlib.metadata

import turbopuffer

# Example 1: the client derives its endpoint from the region and authenticates with a bearer key
assert importlib.metadata.version("turbopuffer") == "3.1.0"
client = turbopuffer.Turbopuffer(api_key="tpuf_test", region="gcp-us-central1")
assert str(client.base_url).rstrip("/") == "https://gcp-us-central1.turbopuffer.com"
assert client.auth_headers == {"Authorization": "Bearer tpuf_test"}, client.auth_headers
assert client.namespace("docs").id == "docs"
print("Example 1 OK:", client.base_url)

## Example 2: Upsert rows: vectors travel as base64 float32

In [ ]:
import base64
import json
import struct

import httpx2
import turbopuffer

seen = []

def api(request):
    seen.append((request.method, request.url.path, json.loads(request.content or b"{}")))
    if request.url.path.endswith("/query"):
        return httpx2.Response(200, json={"rows": [{"id": 2, "$dist": 0.0, "title": "b"}]})
    return httpx2.Response(200, json={"status": "OK", "message": "ok", "rows_affected": 2})

# Example 2: write() sends one POST per namespace; vectors are packed as little-endian float32, base64 encoded
client = turbopuffer.Turbopuffer(api_key="k", region="gcp-us-central1", http_client=httpx2.Client(transport=httpx2.MockTransport(api)))
result = client.namespace("docs").write(upsert_rows=[{"id": 1, "vector": [0.5, 0.25], "title": "a"},
                                                    {"id": 2, "vector": [1.0, 2.0], "title": "b"}],
                                       distance_metric="cosine_distance")
method, path, body = seen[-1]
assert (method, path, result.rows_affected) == ("POST", "/v2/namespaces/docs", 2)
assert struct.unpack("<2f", base64.b64decode(body["upsert_rows"][0]["vector"])) == (0.5, 0.25)
print("Example 2 OK:", body["distance_metric"])

## Example 3: Vector query with a filter

In [ ]:
import json

import httpx2
import turbopuffer

seen = []

def api(request):
    seen.append((request.method, request.url.path, json.loads(request.content or b"{}")))
    if request.url.path.endswith("/query"):
        return httpx2.Response(200, json={"rows": [{"id": 2, "$dist": 0.0, "title": "b"}]})
    return httpx2.Response(200, json={"status": "OK", "message": "ok", "rows_affected": 2})

# Example 3: rank_by, filters and include_attributes are sent as turbopuffer's JSON tuples
client = turbopuffer.Turbopuffer(api_key="k", region="gcp-us-central1", http_client=httpx2.Client(transport=httpx2.MockTransport(api)))
res = client.namespace("docs").query(rank_by=("vector", "ANN", [1.0, 2.0]), top_k=1,
                                     filters=("title", "Eq", "b"), include_attributes=["title"])
assert seen[-1][2] == {"rank_by": ["vector", "ANN", [1.0, 2.0]], "top_k": 1, "filters": ["title", "Eq", "b"],
                       "include_attributes": ["title"]}, seen[-1][2]
assert res.rows[0].id == 2 and res.rows[0].model_extra["title"] == "b"
print("Example 3 OK:", res.rows[0].id)

## Example 4: HTTP errors become typed exceptions

In [ ]:
import httpx2
import turbopuffer

# Example 4: a 404 is NotFoundError, a 401 AuthenticationError (no retries for client errors)
def answer(status):
    transport = httpx2.MockTransport(lambda request: httpx2.Response(status, json={"status": "error", "error": "x"}))
    return turbopuffer.Turbopuffer(api_key="k", region="gcp-us-central1", http_client=httpx2.Client(transport=transport))
for status, error in ((404, turbopuffer.NotFoundError), (401, turbopuffer.AuthenticationError)):
    try:
        answer(status).namespace("missing").query(rank_by=("id", "asc"), top_k=1)
        raise AssertionError("no error")
    except error as caught:
        assert caught.status_code == status
print("Example 4 OK")

## Example 5: The async client

In [ ]:
import json

import httpx2
import turbopuffer

seen = []

def api(request):
    seen.append((request.method, request.url.path, json.loads(request.content or b"{}")))
    if request.url.path.endswith("/query"):
        return httpx2.Response(200, json={"rows": [{"id": 2, "$dist": 0.0, "title": "b"}]})
    return httpx2.Response(200, json={"status": "OK", "message": "ok", "rows_affected": 2})

# Example 5: AsyncTurbopuffer has the same surface; top-level await works in a notebook
client = turbopuffer.AsyncTurbopuffer(api_key="k", region="gcp-us-central1",
                                      http_client=httpx2.AsyncClient(transport=httpx2.MockTransport(api)))
res = await client.namespace("docs").query(rank_by=("id", "asc"), top_k=1)
assert res.rows[0].id == 2 and seen[-1][1] == "/v2/namespaces/docs/query"
await client.close()
print("Example 5 OK")